# Provenance Agent — Quickstart

This notebook walks through the core functions of the provenance agent. Given any Jupyter notebook, the agent identifies which Python libraries are imported and generates an APA bibliography for the software used.

**Functions covered:**
- `parse_notebook` — reads a `.ipynb` file and returns all imported libraries
- `validate_libraries` — checks which requested libraries exist in the notebook
- `extract_libraries` — extracts imports from a single string of Python code
- `strip_ipython_directives` — removes `%magic` and `!shell` lines before parsing
- `collect_entries` — gathers BibTeX entries from YAML and .bib files
- `render_apa` — converts BibTeX to APA format via LLM (Gemini)
- `generate_bibliography` — orchestrates the full bibliography pipeline
- `generate_bibliography_cell` — returns a JSON-formatted markdown cell for PaleoPAL

## Setup

Add `src/` to the path so the module can be imported without installing the package.

In [ ]:
import sys
import numpy
import pandas
sys.path.insert(0, '../src')

from notebook_parser import parse_notebook, extract_libraries, strip_ipython_directives

## 1. `parse_notebook` — scan a full notebook

This is the main function you'll call. Pass a path to any `.ipynb` file and it returns the sorted list of top-level libraries imported anywhere in the notebook. Called with no argument inside a running Jupyter session, it auto-detects the current notebook path (requires `pip install ipynbname`).

In [4]:
result = parse_notebook('sample.ipynb')
print('Libraries found:', result['libraries'])

Libraries found: ['matplotlib', 'numpy', 'pandas', 'pyleoclim']


Automatically parses current notebook with nbformat

In [5]:
result = parse_notebook()
print('Libraries found:', result['libraries'])

Libraries found: ['bibliography', 'notebook_parser', 'numpy', 'pandas', 'sys']


You can also point it at any other notebook by passing an explicit path:

```python
result = parse_notebook('sample.ipynb')
```

## 2. `extract_libraries` — parse a snippet of code

`extract_libraries` works on a raw string rather than a file. It's useful for testing or for processing individual cells. It handles both `import X` and `from X.Y import Z` forms, and always returns the top-level package name (e.g. `matplotlib.pyplot` → `matplotlib`).

In [6]:
code = """
import numpy as np
import pandas as pd
from matplotlib.pyplot import plt
"""
print(extract_libraries(code))

{'pandas', 'matplotlib', 'numpy'}


## 3. `strip_ipython_directives` — clean up magic and shell commands

Paleoclimate notebooks commonly use `%matplotlib inline`, `!pip install ...`, and similar IPython-only syntax. These lines are not valid Python and would cause `ast.parse` to fail. `strip_ipython_directives` removes them so that import extraction can proceed on the rest of the cell.

In [ ]:
raw = """
%matplotlib inline
!pip install pyleoclim
import pyleoclim as pyleo
"""
print(strip_ipython_directives(raw))

The `%` and `!` lines are dropped; the `import` line passes through unchanged. `parse_notebook` calls this automatically on every cell before parsing.

## 4. Dataset Extraction _(coming soon)_

In addition to libraries, the provenance agent will eventually identify datasets loaded in a notebook. Data in these notebooks typically comes from three sources:

- **LiPDGraph** — a graph database queried directly
- **PyLiPD** — local or web-based LiPD files
- **PANGAEA / NOAA NCEI** — datasets fetched via PyleoTUPS

For now, `parse_notebook` returns an empty list as a placeholder.

In [ ]:
result = parse_notebook('workflow.ipynb')
print('Datasets:', result['datasets'])  # TODO: implement dataset extraction

## 5. Full Pipeline Test — parse → collect → render → return string

Test each step of the bibliography pipeline individually:
1. `parse_notebook` — extract libraries from a notebook
2. `validate_libraries` — check which requested libraries are available
3. `collect_entries` — gather BibTeX entries from YAML and .bib files
4. `render_apa` — convert BibTeX to APA via LLM (Gemini)
5. `generate_bibliography` — orchestrate steps 2–4 and handle missing libraries

In [ ]:
from notebook_parser import validate_libraries
from bibliography import collect_entries, render_apa, generate_bibliography, generate_bibliography_cell

In [8]:
# Step 1: Parse this notebook
result = parse_notebook('workflow.ipynb')
print('All libraries found:', result['libraries'])

All libraries found: ['bibliography', 'notebook_parser', 'numpy', 'pandas', 'sys']


In [9]:
# Step 2: Validate a subset of requested libraries
found, not_found = validate_libraries(['pandas', 'numpy', 'fake_lib'], result['libraries'])
print(f'Found: {found}')
print(f'Not found: {not_found}')

Found: ['pandas', 'numpy']
Not found: ['fake_lib']


In [11]:
# Step 3: Collect BibTeX entries
entries = collect_entries(found)
print(f'Collected {len(entries.entries)} BibTeX entries')
entries

Collected 3 BibTeX entries


BibliographyData(
  entries=OrderedCaseInsensitiveDict([
    ('mckinney2010data', Entry('inproceedings',
      fields=[
        ('title', 'Data Structures for Statistical Computing in Python'),
        ('booktitle', 'Proceedings of the 9th Python in Science Conference'),
        ('pages', '56--61'),
        ('year', '2010'),
        ('doi', '10.25080/Majora-92bf1922-00a')],
      persons={'author': [Person('McKinney, Wes')], 'editor': [Person("van der Walt, St\\'efan"), Person('Millman, Jarrod')]})), 
    ('pandas_software', Entry('software',
      fields=[
        ('title', 'pandas-dev/pandas: Pandas'),
        ('year', '2020'),
        ('publisher', 'Zenodo'),
        ('doi', '10.5281/zenodo.3509134'),
        ('url', 'https://doi.org/10.5281/zenodo.3509134')],
      persons={'author': [Person('{The pandas development team}')]})), 
    ('harris2020array', Entry('article',
      fields=[
        ('title', 'Array programming with {NumPy}'),
        ('journal', 'Nature'),
        ('volu

In [ ]:
# Step 4: Render to APA via LLM (Gemini)
apa_text = render_apa(entries)
print(apa_text)

In [ ]:
# Step 5: Generate full bibliography (combines steps 3-4 + not-found placeholders)
bib_text = generate_bibliography(result['libraries'])
print(bib_text)

In [ ]:
# Step 6: Return JSON-formatted markdown cell for the citation
cell_json = generate_bibliography_cell(result['libraries'])
print(cell_json)

---
## Testing on Real Notebooks

The five notebooks below are from the *Comparing Simulated and Reconstructed Climate Variability over the Past Millennium* collection. Each one loads CMIP6/PMIP model output and LMR reconstruction data via `intake`, `xarray`, and `zarr`. Running `parse_notebook` on them exercises the parser against real scientific notebooks.

### C02_b — Data Assimilation with Individual Seasonality

This notebook uses `cfr` (Climate Field Reconstruction), `numpy`, and `pickle`. It exercises the parser on a paleoclimate-specific workflow that includes serialized data loading.

In [ ]:
result = parse_notebook('C02_b_DA_with_individual_seasonality.ipynb')
print(f'Libraries: {result["libraries"]}')
assert result['libraries'] == ['cfr', 'numpy', 'pickle'], f'Unexpected: {result["libraries"]}'
print('PASSED')

In [ ]:
notebooks = [
    'comparing-simulated-reconstructed-climate/CMIP6_LMR.ipynb',
    'comparing-simulated-reconstructed-climate/data_from_esm_cloudcat.ipynb',
    'comparing-simulated-reconstructed-climate/spatial_snapshots_xarray_bonuses.ipynb',
    'comparing-simulated-reconstructed-climate/VICS_dashboard.ipynb',
    'comparing-simulated-reconstructed-climate/widget_primer.ipynb',
]

for nb_path in notebooks:
    result = parse_notebook(nb_path)
    print(f'{nb_path.split("/")[-1]}')
    print(f'  Libraries: {result["libraries"]}')
    print()

## Full Pipeline Test on C02_b

Run the entire bibliography pipeline on a real paleoclimate notebook: parse → validate → collect → render (LLM) → generate.

In [13]:
# Step 1: Parse
result = parse_notebook('C02_b_DA_with_individual_seasonality.ipynb')
print('Libraries found:', result['libraries'])

Libraries found: ['cfr', 'numpy', 'pickle']


In [14]:
# Step 2: Validate
found, not_found = validate_libraries(result['libraries'], result['libraries'])
print(f'Found: {found}')
print(f'Not found: {not_found}')

Found: ['cfr', 'numpy', 'pickle']
Not found: []


In [15]:
# Step 3: Collect BibTeX entries
entries = collect_entries(found)
print(f'Collected {len(entries.entries)} BibTeX entries')

Collected 1 BibTeX entries


In [ ]:
# Step 4: Render to APA via Gemini
apa_text = render_apa(entries)
print(apa_text)

In [16]:
# Step 5: Full bibliography
bib_text = generate_bibliography(result['libraries'])
print(bib_text)

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., ... Oliphant, T. E. (2020). Array programming with NumPy. *Nature, 585*(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

[No citation found for: cfr]
